# Russian Text Style Transfer

See the README for setup and required inputs.

## Validation metrics

In [ ]:
import math
from collections import Counter

def get_word_ngrams(tokens: list, n: int) -> list:
    """
    Разбивает список токенов на n-граммы уровня слов.
    """
    return [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]


def get_char_ngrams(text: str, n: int) -> list:
    """
    Разбивает строку на n-граммы уровня символов.
    """
    return [text[i : i + n] for i in range(len(text) - n + 1)]


In [ ]:

def bleu_single(reference: str, candidate: str, max_n: int = 4) -> float:
    """
    Вычисляет BLEU-score для одной пары (reference, candidate).

    Шаги:
      1. Токенизация (разбиение по пробелам)
      2. Подсчет clipped n-gram precision p_n для каждого n
      3. Вычисление штрафа за краткость BP
      4. Итог: BP * exp(sum(w_n * log(p_n)))

    Args:
        reference  : эталонный текст (строка)
        candidate  : сгенерированный текст (строка)
        max_n      : максимальный порядок n-граммы (по умолчанию 4 -> BLEU-4)

    Returns:
        float в диапазоне [0.0, 1.0]
    """
    ref_tokens = reference.lower().split()
    cand_tokens = candidate.lower().split()

    c = len(cand_tokens)
    r = len(ref_tokens)

    if c == 0:
        return 0.0

    bp = 1.0 if c > r else math.exp(1 - r / c)
    w_n = 1.0 / max_n
    log_avg = 0.0

    for n in range(1, max_n + 1):
        ref_ngrams = Counter(get_word_ngrams(ref_tokens, n))
        cand_ngrams = Counter(get_word_ngrams(cand_tokens, n))

        matches = sum(min(cnt, ref_ngrams[ngram]) for ngram, cnt in cand_ngrams.items())
        total = max(c - n + 1, 1)

        if n == 1 and matches == 0:
            return 0.0

        p_n = (matches + 1) / (total + 1)
        log_avg += w_n * math.log(p_n)

    return bp * math.exp(log_avg)


def chrf_single(reference: str, candidate: str, max_n: int = 6, beta: float = 2.0) -> float:
    """
    Вычисляет chrF-score для одной пары (reference, candidate).

    Шаги:
      1. Разбить оба текста на символьные n-граммы (для n от 1 до max_n)
      2. Для каждого n посчитать Precision и Recall по совпавшим кускам
      3. Усреднить Precision и Recall по всем n
      4. Итог: F_beta-мера из усредненных P и R

    Args:
        reference  : эталонный текст (строка)
        candidate  : сгенерированный текст (строка)
        max_n      : максимальный порядок символьной n-граммы (по умолчанию 6)
        beta       : вес Recall относительно Precision (по умолчанию 2.0 -> chrF2)

    Returns:
        float в диапазоне [0.0, 1.0]
    """
    if len(candidate) == 0 or len(reference) == 0:
        return 0.0

    total_precision = 0.0
    total_recall = 0.0

    for n in range(1, max_n + 1):
        ref_ngrams = Counter(get_char_ngrams(reference, n))
        cand_ngrams = Counter(get_char_ngrams(candidate, n))

        matches = sum(
            min(count, ref_ngrams.get(ngram, 0))
            for ngram, count in cand_ngrams.items()
        )

        total_cand = sum(cand_ngrams.values())
        total_ref = sum(ref_ngrams.values())

        if total_cand > 0:
            total_precision += (matches + 1) / (total_cand + 1)
        if total_ref > 0:
            total_recall += matches / total_ref

    avg_p = total_precision / max_n
    avg_r = total_recall / max_n

    if avg_p + avg_r == 0:
        return 0.0

    return (1 + beta ** 2) * (avg_p * avg_r) / (beta ** 2 * avg_p + avg_r)


In [ ]:

def compute_bleu(y_true: list, y_pred: list, max_n: int = 4) -> float:
    """Считает средний BLEU по всем парам датасета."""
    if not y_true:
        return 0.0

    scores = [bleu_single(ref, cand, max_n) for ref, cand in zip(y_true, y_pred)]
    return sum(scores) / len(scores)


def compute_chrf(y_true: list, y_pred: list, max_n: int = 6, beta: float = 2.0) -> float:
    """Считает средний chrF по всем парам датасета."""
    if not y_true:
        return 0.0

    scores = [chrf_single(ref, cand, max_n, beta) for ref, cand in zip(y_true, y_pred)]
    return sum(scores) / len(scores)


## Prepare paired texts and fine-tune mT5

In [ ]:
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
MODEL_NAME = "google/mt5-small"
OUTPUT_DIR = "mt5_zoomer_baseline_manual"
PACK_MAX_WORDS = 48
MAX_WORDS_TO_KEEP = 160
MAX_SOURCE_LENGTH = 192
MAX_TARGET_LENGTH = 192
MAX_TRAIN_SAMPLES = 12000  # можно поставить None, если хочешь обучаться на всех данных
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
PROMPT_PREFIX = "переведи на современный молодежный сленг: "
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

data_path = Path("dataset.csv")
if not data_path.is_file():
    raise FileNotFoundError("Add a paired Russian text CSV as dataset.csv first.")
raw_df = pd.read_csv(data_path, engine="python")

for column in ["chunk", "translate"]:
    raw_df[column] = raw_df[column].astype(str)
    raw_df[column] = raw_df[column].str.replace("\r\n", "\n", regex=False)
    raw_df[column] = raw_df[column].str.replace("\r", "\n", regex=False)
    raw_df[column] = raw_df[column].str.replace(r"[ \t]+", " ", regex=True)
    raw_df[column] = raw_df[column].str.replace(r"\n{3,}", "\n\n", regex=True)
    raw_df[column] = raw_df[column].str.strip()

raw_df = raw_df[(raw_df["chunk"] != "") & (raw_df["translate"] != "")]
raw_df = raw_df.drop_duplicates().reset_index(drop=True)

raw_df["src_words"] = raw_df["chunk"].str.split().str.len()
raw_df["tgt_words"] = raw_df["translate"].str.split().str.len()
print(f"Исходный датасет после очистки: {len(raw_df)} пар")
print(
    "Средняя длина исходных текстов (слов):",
    round(raw_df["src_words"].mean(), 1),
    round(raw_df["tgt_words"].mean(), 1),
)
display(raw_df[["chunk", "translate"]].head(2))

sentence_boundary = re.compile(r"(?<=[.!?…])\s+")
expanded_rows = []

for row in raw_df.itertuples(index=False):
    source_text = row.chunk
    target_text = row.translate

    source_paragraphs = [p.strip() for p in re.split(r"\n\s*\n", source_text) if p.strip()]
    target_paragraphs = [p.strip() for p in re.split(r"\n\s*\n", target_text) if p.strip()]

    if len(source_paragraphs) == len(target_paragraphs) and len(source_paragraphs) > 1:
        source_units = source_paragraphs
        target_units = target_paragraphs
    else:
        source_sentences = [
            s.strip() for s in sentence_boundary.split(source_text.replace("\n", " ")) if s.strip()
        ]
        target_sentences = [
            s.strip() for s in sentence_boundary.split(target_text.replace("\n", " ")) if s.strip()
        ]

        if len(source_sentences) == len(target_sentences) and len(source_sentences) > 1:
            source_units = source_sentences
            target_units = target_sentences
        else:
            source_units = [source_text]
            target_units = [target_text]

    current_source = []
    current_target = []
    current_words = 0

    for source_piece, target_piece in zip(source_units, target_units):
        piece_words = max(len(source_piece.split()), len(target_piece.split()))

        if current_source and current_words + piece_words > PACK_MAX_WORDS:
            expanded_rows.append(
                {
                    "chunk": " ".join(current_source).strip(),
                    "translate": " ".join(current_target).strip(),
                }
            )
            current_source = []
            current_target = []
            current_words = 0

        current_source.append(source_piece)
        current_target.append(target_piece)
        current_words += piece_words

    if current_source:
        expanded_rows.append(
            {
                "chunk": " ".join(current_source).strip(),
                "translate": " ".join(current_target).strip(),
            }
        )

prepared_df = pd.DataFrame(expanded_rows).drop_duplicates().reset_index(drop=True)
prepared_df["src_words"] = prepared_df["chunk"].str.split().str.len()
prepared_df["tgt_words"] = prepared_df["translate"].str.split().str.len()
prepared_df = prepared_df[
    (prepared_df["src_words"] <= MAX_WORDS_TO_KEEP)
    & (prepared_df["tgt_words"] <= MAX_WORDS_TO_KEEP)
].reset_index(drop=True)

print(f"После разбиения и фильтрации: {len(prepared_df)} пар")
print(
    "Средняя длина подготовленных текстов (слов):",
    round(prepared_df["src_words"].mean(), 1),
    round(prepared_df["tgt_words"].mean(), 1),
)
display(prepared_df[["chunk", "translate", "src_words", "tgt_words"]].sample(3, random_state=SEED))

train_df, val_df = train_test_split(
    prepared_df[["chunk", "translate"]],
    test_size=0.1,
    random_state=SEED,
    shuffle=True,
)

if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(train_df):
    train_df = train_df.sample(MAX_TRAIN_SAMPLES, random_state=SEED)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print("Device:", DEVICE)

class TextPairDataset(Dataset):
    def __init__(self, frame):
        self.sources = frame["chunk"].tolist()
        self.targets = frame["translate"].tolist()

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        return {
            "source": self.sources[idx],
            "target": self.targets[idx],
        }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

def collate_fn(batch):
    sources = [PROMPT_PREFIX + item["source"] for item in batch]
    targets = [item["target"] for item in batch]

    model_inputs = tokenizer(
        sources,
        padding=True,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_tensors="pt",
    )
    labels = tokenizer(
        text_target=targets,
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt",
    )

    labels_ids = labels["input_ids"]
    labels_ids[labels_ids == tokenizer.pad_token_id] = -100
    model_inputs["labels"] = labels_ids
    return model_inputs

train_loader = DataLoader(
    TextPairDataset(train_df),
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    TextPairDataset(val_df),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
best_metrics = None
best_model_path = os.path.join(OUTPUT_DIR, "best_model.pt")
os.makedirs(OUTPUT_DIR, exist_ok=True)
training_history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_losses = []
    train_progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [train]")

    for step, batch in enumerate(train_progress, start=1):
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        (loss / GRADIENT_ACCUMULATION_STEPS).backward()
        train_losses.append(float(loss.item()))

        if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        train_progress.set_postfix(loss=round(np.mean(train_losses[-20:]), 4))

    model.eval()
    val_losses = []
    predictions = []
    references = []
    eval_progress = tqdm(val_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [eval]")

    for batch in eval_progress:
        labels_for_decode = batch["labels"].clone()
        batch = {key: value.to(DEVICE) for key, value in batch.items()}

        with torch.no_grad():
            outputs = model(**batch)
            generated_ids = model.generate(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=1.0,
                early_stopping=True,
            )

        val_losses.append(float(outputs.loss.item()))

        predicted_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        labels_for_decode[labels_for_decode == -100] = tokenizer.pad_token_id
        target_texts = tokenizer.batch_decode(labels_for_decode, skip_special_tokens=True)

        predicted_texts = [re.sub(r"\s+", " ", text).strip() for text in predicted_texts]
        target_texts = [re.sub(r"\s+", " ", text).strip() for text in target_texts]

        predictions.extend(predicted_texts)
        references.extend(target_texts)

    epoch_metrics = {
        "epoch": epoch + 1,
        "train_loss": float(np.mean(train_losses)),
        "eval_loss": float(np.mean(val_losses)),
        "eval_bleu": float(compute_bleu(references, predictions)),
        "eval_chrf": float(compute_chrf(references, predictions)),
    }
    training_history.append(epoch_metrics)
    print(epoch_metrics)

    if best_metrics is None or epoch_metrics["eval_chrf"] > best_metrics["eval_chrf"]:
        best_metrics = epoch_metrics
        torch.save(model.state_dict(), best_model_path)

model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

training_history = pd.DataFrame(training_history)
display(training_history)
print("Лучшие валидационные метрики:", best_metrics)


## Generate and evaluate rewritten text

In [ ]:
# Функция для перевода текста на основе обученной тобою модели
model.eval()
model.config.use_cache = True


def translate(old_text):
    old_text = str(old_text).replace("\r\n", "\n").replace("\r", "\n")
    old_text = re.sub(r"[ \t]+", " ", old_text)
    old_text = re.sub(r"\n{3,}", "\n\n", old_text).strip()

    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", old_text) if p.strip()]
    if not paragraphs and old_text:
        paragraphs = [old_text]

    prepared_chunks = []
    for paragraph in paragraphs:
        sentences = [s.strip() for s in sentence_boundary.split(paragraph.replace("\n", " ")) if s.strip()]

        if len(sentences) <= 1:
            prepared_chunks.append(paragraph)
            continue

        current_chunk = []
        current_words = 0
        for sentence in sentences:
            sentence_words = len(sentence.split())
            if current_chunk and current_words + sentence_words > PACK_MAX_WORDS:
                prepared_chunks.append(" ".join(current_chunk).strip())
                current_chunk = []
                current_words = 0
            current_chunk.append(sentence)
            current_words += sentence_words

        if current_chunk:
            prepared_chunks.append(" ".join(current_chunk).strip())

    if not prepared_chunks:
        return ""

    batch = tokenizer(
        [PROMPT_PREFIX + text for text in prepared_chunks],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        generated_ids = model.generate(
            **batch,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=4,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            early_stopping=True,
        )

    translated_chunks = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    translated_chunks = [re.sub(r"\s+", " ", text).strip() for text in translated_chunks]
    return "\n\n".join(translated_chunks)


In [ ]:
# Посчитай среднее значение метрик и функции потерь на объектах из validation-части датасета
cross_entropy_loss = float(best_metrics["eval_loss"])
chr_f_metric = float(best_metrics["eval_chrf"])
bleu_metric = float(best_metrics["eval_bleu"])

print(
    {
        "cross_entropy_loss": round(cross_entropy_loss, 4),
        "chrF": round(chr_f_metric, 4),
        "BLEU": round(bleu_metric, 4),
    }
)


In [ ]:
# Пример изначального текста и его переведённая версия
example = '''Я вас любил: любовь еще, быть может,
В душе моей угасла не совсем'''

translated_example = translate(example)

print(f"Кросс-энтропия: {cross_entropy_loss}\nChr-F: {chr_f_metric}\nBLEU: {bleu_metric}\n\nПеревод: {translated_example}")